# Buổi 21 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `tai_chinh.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Sự thật cách điệu (mục 4.2)

In [ ]:
%matplotlib inline
import warnings

import numpy as np
import pandas as pd
import tai_chinh as tc

warnings.simplefilter("ignore")
gia, r, rv = tc.btc_ngay()
jpy = tc.loi_suat(tc.doc_jpy())
print(pd.DataFrame({"BTC": tc.su_that_cach_dieu(r), "JPY/USD": tc.su_that_cach_dieu(jpy)}).round(4).to_string())

## Bước 2 — Demo đoán giá (mục 4.3)

Sửa `du_bao_gia` và `danh_gia_gia` rồi chạy lại ô này.

In [ ]:
kq = tc.du_bao_gia(gia)
print("số ngày chấm:", len(kq), "| từ", kq.index.min().date(), "tới", kq.index.max().date())
print({k: round(v, 4) for k, v in tc.danh_gia_gia(kq).items()})

## Bước 3 — GARCH và HAR (mục 4.4–4.5)

In [ ]:
har = tc.du_bao_har(rv, tc.MOC)
y = rv.loc[har.index]
du_bao = {"HAR": har,
          "GARCH-t": tc.du_bao_garch(r, tc.MOC)["phương sai"].reindex(har.index),
          "GJR-t": tc.du_bao_garch(r, tc.MOC, o=1)["phương sai"].reindex(har.index),
          "cố định": pd.Series(r[r.index < pd.Timestamp(tc.MOC)].var(), index=har.index)}
print(pd.DataFrame({ten: {"QLIKE": tc.qlike(y, h), "MSE": float(np.mean((y - h) ** 2))} for ten, h in du_bao.items()}).T.round(3))

## Bước 4 — VaR và Kupiec (mục 4.6)

Sửa `var_99` rồi chạy lại ô này.

In [ ]:
print(tc.kupiec(jpy, tc.var_99(jpy)))

## Bước 5 — Markov switching (hộp "Nâng cao" mục 4.6)

Sau đó trong terminal ở thư mục `lab/`: `python lab.py check` — phải xanh 7/7.

In [ ]:
p = tc.markov_2_trang_thai(jpy)
print("tỷ lệ ngày ở trạng thái bất ổn:", round(float((p > 0.5).mean()), 3))
print(p.resample("YS").mean().round(2).set_axis(p.resample("YS").mean().index.year).to_string())